# Pista A — Motor empírico de simulación (ECO | Wind)

Sandbox de la **Fase 1 / Pista A** del plan técnico (`plan-tecnico-eco-wind.md`).
Objetivo: un pipeline trazable `simular(lat, lon, altura_buje, modelo, N) -> kWh_anual`
usando el catálogo de Flower Turbines ya validado (`engine/flower_turbines_curves.py`).

**Nota sobre entornos de ejecución:** este notebook se escribió y probó en un sandbox de
Claude Code sin salida de red a `power.larc.nasa.gov` (política de egress del entorno).
La celda de NASA POWER está armada para degradar sola a datos sintéticos cuando eso pasa,
así que el notebook corre de punta a punta en cualquier entorno — pero el resultado real
(con viento real del sitio) solo sale corriéndolo en **Google Colab** o cualquier entorno
con internet normal. Los pasos 3 y 4 (corrección de altura, ensamblado) sí están validados
acá con datos sintéticos.


In [1]:
import os

if os.path.exists("../engine/flower_turbines_curves.py"):
    # Ya estamos dentro de una copia clonada del repo (p.ej. este sandbox de desarrollo)
    print("Repo ya presente -- no hace falta clonar.")
else:
    # Runtime de Colab: clonar si es la primera vez, o traer lo ultimo si el
    # runtime ya tenia una copia de una corrida anterior (evita quedar con
    # una version vieja del repo -- p.ej. sin datos_clima/ -- entre corridas
    # de la misma sesion).
    if os.path.exists("/content/ECO-Wind"):
        get_ipython().system("git -C /content/ECO-Wind pull")
    else:
        get_ipython().system("git clone https://github.com/Sogo2012/ECO-Wind.git /content/ECO-Wind")
    get_ipython().run_line_magic("cd", "/content/ECO-Wind/notebooks")


Repo ya presente -- no hace falta clonar.


## Paso 1 — Módulo base (`engine/flower_turbines_curves.py`)

In [2]:
import sys
sys.path.insert(0, "..")

import calendar
import numpy as np
import pandas as pd
import requests

from engine.flower_turbines_curves import (
    CURVE_COEFFICIENTS,
    power_isolated,
    bouquet_multiplier,
    power_in_bouquet,
)

print("Modelos disponibles:", list(CURVE_COEFFICIENTS))
print(f"Medium Tulip @ 12 m/s, aislada: {float(power_isolated(12, 'medium_tulip')):.1f} W")


Modelos disponibles: ['small_tulip', 'medium_tulip', 'three_m_tulip', 'large_tulip', 'al13_2m', 'al13_4m', 'al13_6m', 'al13_8m']
Medium Tulip @ 12 m/s, aislada: 622.2 W


## Paso 2 — Ingesta climática (NASA POWER Hourly)

Punto `community=SB`, parámetros `WS10M`/`WS50M`/`T2M`, formato JSON, año completo
(8,760 h ó 8,784 h en bisiesto).


In [3]:
NASA_POWER_HOURLY_URL = "https://power.larc.nasa.gov/api/temporal/hourly/point"


def fetch_nasa_power_hourly(lat, lon, year, community="SB",
                             parameters=("WS10M", "WS50M", "T2M")):
    """
    Descarga viento/temperatura horarios de NASA POWER para un año completo
    en una coordenada arbitraria. Devuelve un DataFrame con índice datetime
    horario y una columna por parámetro.

    NO EJECUTADO CON ÉXITO EN EL SANDBOX DE DESARROLLO: sin salida de red a
    power.larc.nasa.gov ahí. Validar en Colab (o cualquier entorno con
    internet normal) antes de confiar en el resultado.
    """
    params = {
        "parameters": ",".join(parameters),
        "community": community,
        "longitude": lon,
        "latitude": lat,
        "start": f"{year}0101",
        "end": f"{year}1231",
        "format": "JSON",
    }
    resp = requests.get(NASA_POWER_HOURLY_URL, params=params, timeout=60)
    resp.raise_for_status()
    param_data = resp.json()["properties"]["parameter"]
    df = pd.DataFrame(param_data)
    df.index = pd.to_datetime(df.index, format="%Y%m%d%H")
    df.index.name = "datetime"

    horas_esperadas = 8784 if calendar.isleap(year) else 8760
    if len(df) != horas_esperadas:
        raise ValueError(f"Esperaba {horas_esperadas} horas, llegaron {len(df)}")
    return df


In [4]:
def generar_clima_sintetico(year=2023, v_media=3.5, seed=42):
    """
    SOLO PARA PROBAR EL PIPELINE SIN RED. Serie horaria sintética con forma
    estacional simple (pico ilustrativo dic-abr, tipo estación seca de Costa
    Rica) + ruido Weibull. v_media=3.5 m/s es un valor ilustrativo, NO viene
    de ninguna fuente medida -- reemplazar por fetch_nasa_power_hourly() con
    dato real antes de sacar cualquier conclusión de negocio.
    """
    n_horas = 8784 if calendar.isleap(year) else 8760
    idx = pd.date_range(f"{year}-01-01", periods=n_horas, freq="h")
    rng = np.random.default_rng(seed)
    estacional = 1.0 + 0.4 * np.cos(2 * np.pi * (idx.dayofyear - 30) / 365)
    ruido = rng.weibull(2.0, size=n_horas)
    ws10m = np.clip(v_media * estacional * ruido / ruido.mean(), 0, None)
    return pd.DataFrame(
        {"WS10M": ws10m, "WS50M": ws10m * 1.15, "T2M": 22.0}, index=idx
    )


In [5]:
# Coordenada de prueba: Aeropuerto Juan Santamaría (misma zona que el EPW de
# datos_clima/, para comparar NASA POWER vs. estación manzanas con manzanas)
LAT, LON, YEAR = 10.00342327565566, -84.20332993360161, 2023

try:
    df_clima = fetch_nasa_power_hourly(LAT, LON, YEAR)
    print(f"OK -- {len(df_clima)} horas reales descargadas de NASA POWER.")
    datos_reales = True
except Exception as exc:
    print(f"No se pudo descargar de NASA POWER en este entorno ({exc!r}).")
    print("Sigo con datos SINTÉTICOS solo para probar el resto del pipeline.")
    df_clima = generar_clima_sintetico(YEAR, v_media=3.5)
    datos_reales = False

df_clima.head()


No se pudo descargar de NASA POWER en este entorno (ProxyError(MaxRetryError("HTTPSConnectionPool(host='power.larc.nasa.gov', port=443): Max retries exceeded with url: /api/temporal/hourly/point?parameters=WS10M%2CWS50M%2CT2M&community=SB&longitude=-84.20332993360161&latitude=10.00342327565566&start=20230101&end=20231231&format=JSON (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"))).
Sigo con datos SINTÉTICOS solo para probar el resto del pipeline.


,WS10M,WS50M,T2M
2023-01-01 00:00:00,8.325665,9.574514,22.0
2023-01-01 01:00:00,8.207046,9.438103,22.0
2023-01-01 02:00:00,8.291923,9.535712,22.0
2023-01-01 03:00:00,2.840222,3.266255,22.0
2023-01-01 04:00:00,1.578642,1.815438,22.0


## Paso 2b — Fuente alternativa: EPW de estación real (offline)

Contraste con un archivo EPW/TMYx real (15 años de la estación del Aeropuerto Juan
Santamaría, `climate.onebuilding.org`, en `datos_clima/`). A diferencia de NASA POWER,
esto **no depende de la red** — funciona igual en este sandbox, en Colab, o donde sea —
y es un promedio de observaciones reales de 15 años en vez de una celda de reanálisis
satelital de ~50-60 km. La columna de viento del EPW ya está a 10m sobre el suelo
(misma referencia que WS10M), así que entra directo al mismo pipeline.

In [6]:
def load_epw_wind(path, year=2023):
    """
    Carga la velocidad de viento horaria de un archivo EPW (EnergyPlus
    Weather / TMYx, típicamente de climate.onebuilding.org).

    La columna 22 del EPW (índice 21) es Wind Speed a 10m sobre el suelo --
    misma altura de referencia que WS10M de NASA POWER, compatible directo
    con wind_at_height()/simular().

    year: año "etiqueta" para el índice datetime (un TMYx mezcla meses de
    años reales distintos -- no afecta promedios anuales/mensuales).
    """
    df = pd.read_csv(path, skiprows=8, header=None)
    idx = pd.date_range(f"{year}-01-01", periods=len(df), freq="h")
    return pd.DataFrame({"WS10M": df[21].values, "T2M": df[6].values}, index=idx)


EPW_PATH = "../datos_clima/CRI_AL_San.Jose-Santamaria.Intl.AP.787620_TMYx.2007-2021.epw"
df_epw = load_epw_wind(EPW_PATH)

fuente_actual = "NASA POWER real" if datos_reales else "NASA POWER (sintético, fallback -- sin red en este entorno)"
print(f"Media anual WS10M -- {fuente_actual}: {df_clima['WS10M'].mean():.2f} m/s")
print(f"Media anual WS10M -- EPW estación (15 años, aeropuerto): {df_epw['WS10M'].mean():.2f} m/s")
print(f"Horas con WS10M < cut-in (0.7 m/s) -- {fuente_actual}: {(df_clima['WS10M'] < 0.7).mean()*100:.1f}%")
print(f"Horas con WS10M < cut-in (0.7 m/s) -- EPW estación: {(df_epw['WS10M'] < 0.7).mean()*100:.1f}%")


Media anual WS10M -- NASA POWER (sintético, fallback -- sin red en este entorno): 3.51 m/s
Media anual WS10M -- EPW estación (15 años, aeropuerto): 4.03 m/s
Horas con WS10M < cut-in (0.7 m/s) -- NASA POWER (sintético, fallback -- sin red en este entorno): 4.4%
Horas con WS10M < cut-in (0.7 m/s) -- EPW estación: 3.3%


## Paso 2c — Fuente estadística: distribución de Weibull (Global Wind Atlas)

Para sitios sin EPW/estación cercana, la propuesta (investigación de Pablo, validada contra lo que encontré del GWA) es no usar NASA POWER crudo -- reemplazarlo por los parámetros de Weibull ($A$=escala, $k$=forma) que el Global Wind Atlas reporta para una coordenada exacta a una altura dada. El GWA hace *downscaling* microescala a 250m sobre topografía real (a diferencia de la rejilla de ~50km de NASA POWER/MERRA-2), así que sí captura canalización orográfica, aceleración en lomas/cañones, etc.

**Diferencia clave con las fuentes anteriores:** esto reconstruye la *distribución* estadística correcta del viento (buena para kWh/año y kWh/mes), pero **no tiene estructura temporal real** -- no sabe qué hora del día o qué día del año sopla más. Para lo anual/mensual alcanza; para correlacionar contra una curva de demanda horaria de un edificio (Fase 2) hace falta la Pista C de abajo (ERA5 + corrección de sesgo).

**Pendiente de datos reales:** todavía no tengo los A/k reales del GWA para el aeropuerto -- globalwindatlas.info está bloqueado desde este sandbox, igual que NASA POWER. Los valores de A/k de la celda de abajo son ilustrativos (inventados para probar que el código funciona), NO son del GWA real. Hace falta que alguien con internet (vos, o yo en Colab) saque el punto del mapa de GWA para el sitio y me pase A y k reales.

In [7]:
import calendar
from scipy.stats import weibull_min


def generar_clima_weibull(A, k, year=2023, seed=42):
    """
    Serie horaria de viento sintética a partir de los parámetros de Weibull
    (A=escala en m/s, k=forma, adimensional) que reporta el Global Wind
    Atlas para un punto y altura de referencia dados.

    IMPORTANTE -- esto reconstruye la distribución de largo plazo, NO una
    serie cronológica real: cada hora es un sorteo independiente de la
    misma Weibull(A, k), sin ciclo diurno, sin estacionalidad, sin
    autocorrelación día a día. Sirve para kWh/año o kWh/mes agregados;
    NO sirve para correlacionar generación horaria contra demanda horaria
    de un edificio -- para eso hace falta la Pista C (ERA5 + bias
    correction), todavía no implementada acá.

    A, k : parámetros de Weibull del GWA para el punto y la altura de
           interés (el GWA los da a 10/50/100/150/200m -- pasar a
           wind_at_height() el mismo h_ref con el que se sacaron A/k).
    """
    n_horas = 8784 if calendar.isleap(year) else 8760
    idx = pd.date_range(f"{year}-01-01", periods=n_horas, freq="h")
    rng = np.random.default_rng(seed)
    ws = weibull_min.rvs(k, scale=A, size=n_horas, random_state=rng)
    return pd.DataFrame({"WS10M": ws, "T2M": 22.0}, index=idx)


# Valores ILUSTRATIVOS -- NO son del GWA real, solo prueban que el código
# corre. Reemplazar por A/k reales del punto del aeropuerto en cuanto los
# tengamos (GWA a 10m, para que sea comparable con WS10M de las otras
# fuentes sin tener que cambiar h_ref en wind_at_height).
A_ilustrativo, k_ilustrativo = 4.2, 2.0
df_weibull = generar_clima_weibull(A_ilustrativo, k_ilustrativo)

print(f"Media teórica de Weibull(A={A_ilustrativo}, k={k_ilustrativo}): "
      f"{weibull_min.mean(k_ilustrativo, scale=A_ilustrativo):.2f} m/s")
print(f"Media de la muestra generada (8760 horas): {df_weibull['WS10M'].mean():.2f} m/s")


Media teórica de Weibull(A=4.2, k=2.0): 3.72 m/s
Media de la muestra generada (8760 horas): 3.70 m/s


## Paso 3 — Corrección de altura (perfil logarítmico)

In [8]:
def wind_at_height(v_ref, h_ref, h_target, z0=0.3):
    """
    Perfil logarítmico de viento: v(h) = v_ref * ln(h_target/z0) / ln(h_ref/z0)

    h_ref   : altura del dato de referencia (10 para WS10M, 50 para WS50M)
    h_target: altura real de buje de la turbina
    z0      : longitud de rugosidad (m). Default 0.3 = suburbano/urbano bajo.
              Ajustar por sitio: 0.03 campo abierto, 0.1 cultivos bajos,
              0.3 suburbano (default), 1.0 urbano denso.

    Las turbinas Flower Turbines son muy bajas (buje entre 1 y 6 m), casi
    siempre POR DEBAJO de los 10 m de referencia -- a diferencia de una HAWT
    grande (buje 80m+), acá la corrección casi siempre REDUCE la velocidad
    respecto al dato crudo de NASA POWER.
    """
    v_ref = np.asarray(v_ref, dtype=float)
    return v_ref * np.log(h_target / z0) / np.log(h_ref / z0)


# Autotest con valores sintéticos (no depende de la red)
print("Autotest perfil logarítmico, v_10m = 5.00 m/s:")
for h in [1.4, 3.0, 6.0, 10.0]:
    print(f"  altura buje={h:4.1f} m  ->  v={wind_at_height(5.0, 10, h):.2f} m/s")
assert abs(wind_at_height(5.0, 10, 10) - 5.0) < 1e-9, "en h_target=h_ref debe dar v_ref exacto"
print("OK: en h_target = h_ref da v_ref exacto.")


Autotest perfil logarítmico, v_10m = 5.00 m/s:
  altura buje= 1.4 m  ->  v=2.20 m/s
  altura buje= 3.0 m  ->  v=3.28 m/s
  altura buje= 6.0 m  ->  v=4.27 m/s
  altura buje=10.0 m  ->  v=5.00 m/s
OK: en h_target = h_ref da v_ref exacto.


## Paso 4 — Ensamblar `simular(lat, lon, altura_buje, modelo, N) -> kWh_anual`

In [9]:
def simular(df_clima, altura_buje, modelo, N, h_ref=10, z0=0.3, metodo_bouquet="real"):
    """
    Ensambla la serie horaria de potencia del clúster y la agrega a kWh
    mensual/anual, usando P(v) = k*v^3 x M(N) del motor empírico.

    df_clima: DataFrame con índice datetime horario y columna 'WS10M' (m/s)
              -- viene de fetch_nasa_power_hourly() o de datos sintéticos;
              la función no sabe ni le importa el origen.
    """
    v_hub = wind_at_height(df_clima["WS10M"].values, h_ref, altura_buje, z0=z0)
    potencia_w_por_turbina = power_in_bouquet(v_hub, modelo, N, metodo=metodo_bouquet)

    serie = pd.Series(potencia_w_por_turbina, index=df_clima.index,
                       name="potencia_W_por_turbina")
    energia_cluster_kwh = serie * N / 1000.0
    return {
        "serie_horaria_W_por_turbina": serie,
        "kwh_mensual": energia_cluster_kwh.resample("MS").sum(),
        "kwh_anual": float(energia_cluster_kwh.sum()),
    }


resultado = simular(df_clima, altura_buje=3.0, modelo="medium_tulip", N=3)
etiqueta = "REAL (NASA POWER)" if datos_reales else "SINTÉTICO -- no es dato real del sitio"
print(f"Escenario: Medium Tulip x3 (bouquet), buje a 3.0 m, San José CR, {YEAR} [{etiqueta}]")
print(f"kWh/año: {resultado['kwh_anual']:.1f}")
print()
print(resultado["kwh_mensual"])


Escenario: Medium Tulip x3 (bouquet), buje a 3.0 m, San José CR, 2023 [SINTÉTICO -- no es dato real del sitio]
kWh/año: 426.6

2023-01-01    78.248672
2023-02-01    69.182152
2023-03-01    64.546197
2023-04-01    38.412973
2023-05-01    21.077526
2023-06-01     9.701907
2023-07-01     6.164773
2023-08-01     6.898122
2023-09-01    10.271701
2023-10-01    20.338143
2023-11-01    36.439241
2023-12-01    65.307598
Freq: MS, Name: potencia_W_por_turbina, dtype: float64


In [10]:
print(f"{'Buje (m)':>10}  {'v_hub medio (m/s)':>18}  {'% horas<cutin':>14}  {'kWh/año':>10}")
for altura in [1.4, 3.0, 6.0]:
    r_epw = simular(df_epw, altura_buje=altura, modelo="medium_tulip", N=3)
    v_hub_epw = wind_at_height(df_epw["WS10M"].values, 10, altura, z0=0.3)
    pct = (v_hub_epw < 0.7).mean() * 100
    print(f"{altura:>10.1f}  {v_hub_epw.mean():>18.2f}  {pct:>13.1f}%  {r_epw['kwh_anual']:>10.1f}")

print()
print("(Mismo escenario que arriba: Medium Tulip x3 en bouquet, z0=0.3.)")
print(f"Para comparar: con [{etiqueta}] a 3.0m dio {resultado['kwh_anual']:.1f} kWh/año.")


  Buje (m)   v_hub medio (m/s)   % horas<cutin     kWh/año
       1.4                1.77           19.8%       181.3
       3.0                2.65           10.2%       606.8
       6.0                3.45            3.4%      1337.0

(Mismo escenario que arriba: Medium Tulip x3 en bouquet, z0=0.3.)
Para comparar: con [SINTÉTICO -- no es dato real del sitio] a 3.0m dio 426.6 kWh/año.


In [11]:
r_weibull = simular(df_weibull, altura_buje=3.0, modelo="medium_tulip", N=3)
print(f"kWh/año [Weibull ILUSTRATIVO A={A_ilustrativo}, k={k_ilustrativo}, buje 3.0m]: "
      f"{r_weibull['kwh_anual']:.1f}")
print("(Con A/k reales del GWA este número va a cambiar -- estos A/k son inventados.)")


kWh/año [Weibull ILUSTRATIVO A=4.2, k=2.0, buje 3.0m]: 400.3
(Con A/k reales del GWA este número va a cambiar -- estos A/k son inventados.)


## Paso 5 — Sanity check de orden de magnitud

In [12]:
# Referencia externa (Kilowatts UK, distribuidor UK): 1,000-5,000 kWh/año
# típico para turbinas pequeñas. No es para copiar el número -- Costa Rica
# tiene otro recurso eólico y contexto -- solo para confirmar que el orden
# de magnitud del resultado no es absurdo.
REF_UK_BAJO, REF_UK_ALTO = 1000, 5000
kwh = resultado["kwh_anual"]

print(f"Resultado [{etiqueta}]: {kwh:.0f} kWh/año")
print(f"Referencia UK (orden de magnitud, turbinas pequeñas): {REF_UK_BAJO}-{REF_UK_ALTO} kWh/año")

if REF_UK_BAJO * 0.3 <= kwh <= REF_UK_ALTO * 3:
    print("-> Orden de magnitud razonable.")
else:
    print("-> FUERA de rango incluso con margen amplio -- revisar el pipeline antes de confiar en él.")

if not datos_reales:
    print()
    print("!! Este sanity check corrió sobre datos SINTÉTICOS. Repetir en Colab con NASA POWER")
    print("   real antes de sacar cualquier conclusión sobre el recurso eólico de Costa Rica.")


Resultado [SINTÉTICO -- no es dato real del sitio]: 427 kWh/año
Referencia UK (orden de magnitud, turbinas pequeñas): 1000-5000 kWh/año
-> Orden de magnitud razonable.

!! Este sanity check corrió sobre datos SINTÉTICOS. Repetir en Colab con NASA POWER
   real antes de sacar cualquier conclusión sobre el recurso eólico de Costa Rica.


## Resumen y próximos pasos

- **Validado en este sandbox, de punta a punta y con datos reales:** `engine/flower_turbines_curves.py` (Paso 1), corrección de altura (Paso 3, autotest), `simular()` (Paso 4), contra un **EPW de estación real** (Paso 2b, offline), y el generador **Weibull/GWA** (Paso 2c) probado con A/k ilustrativos (código funciona; faltan A/k reales).
- **Hallazgo confirmado (misma coordenada -- Aeropuerto Juan Santamaría):** NASA POWER da 1.30 m/s de viento medio anual a 10m, contra 4.03 m/s de la estación real (15 años, TMYx) -- ~3.1x en la media, ~37x en energía anual por la relación cúbica de potencia. Diagnóstico (investigación de Pablo, consistente con lo que yo encontré): la rejilla de ~50km de MERRA-2 (motor detrás de NASA POWER) promedia/aplana la topografía del Valle Central y es ciega a la canalización orográfica que acelera el viento en el aeropuerto. El EPW ancla a observaciones reales (ISD) con ERA5 (rejilla más fina, ~31km) solo para rellenar huecos.
- **Ruta hacia adelante para sitios sin EPW cercano (propuesta de Pablo, en implementación):**
  1. ~~NASA POWER crudo~~ -- descartado como fuente primaria, confirmado con evidencia.
  2. **Global Wind Atlas (GWA)** -- downscaling a 250m sobre topografía real (WAsP, DTU). Da parámetros de Weibull (A, k) por punto y altura. Implementado en Paso 2c (`generar_clima_weibull`), pero **con A/k ilustrativos, no reales todavía** -- globalwindatlas.info está bloqueado desde este sandbox igual que NASA POWER; hace falta sacar el punto real de la web (o desde Colab) para el sitio de interés.
  3. **ERA5 + corrección de sesgo (quantile mapping)** para cuando haga falta estructura horaria real (correlacionar contra demanda de un edificio, Fase 2) -- **todavía no implementado**. Requiere cuenta + API key de Copernicus CDS (`cdsapi`), también bloqueado desde este sandbox; se puede intentar desde Colab.
- **Pendiente:** conseguir A/k reales del GWA para el aeropuerto (y para los sitios reales de proyectos), y decidir si vale la pena construir la Pista C (ERA5 + bias correction) ahora o más adelante.
- **Variables abiertas** (sección 4 del plan, no bloqueantes): calibración K(v) contra datos de campo reales, z0 real del sitio (acá se usó 0.3 como default ilustrativo), validez del M(N) exponencial más allá de N=10 o en layouts 2D.
